In [16]:
!pip install -q transformers accelerate

In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [18]:
def ask_model(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    answer = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        answer,
        skip_special_tokens=True
    )

In [19]:
zero_shot_prompt = """
Explain why regular exercise is beneficial.
Give 3 simple reasons.
"""

zero_shot_answer = ask_model(zero_shot_prompt)

print("ZERO-SHOT OUTPUT")
print("=" * 50)
print(zero_shot_answer)

ZERO-SHOT OUTPUT
Regular exercise is beneficial for several reasons:

1. **Improved Physical Health**: Regular physical activity helps maintain and improve overall physical health. It strengthens muscles, bones, and joints, reduces the risk of chronic diseases such as heart disease, diabetes, and certain types of cancer, and enhances cardiovascular fitness.

2. **Enhanced Mental Health**: Exercise has been shown to have a positive impact on mental health. It can reduce symptoms of depression and anxiety, increase feelings of well-being, and improve mood. Additionally, it can help manage stress levels through various physiological responses like increased endorphins and improved sleep quality.

3. **Increased Energy Levels**: Engaging in regular physical activity can boost your energy levels. This is because exercise increases the production of hormones that


In [20]:
few_shot_prompt = """
Example 1:

Question: Why is drinking water important?

Answer:
1. It keeps the body hydrated.
2. It supports normal body functions.
3. It helps regulate body temperature.


Example 2:

Question: Why is sleeping important?

Answer:
1. It helps the body recover.
2. It improves concentration.
3. It supports overall health.


Now answer this question:

Question: Why is regular exercise beneficial?

Answer:
"""

few_shot_answer = ask_model(few_shot_prompt)

print("FEW-SHOT OUTPUT")
print("=" * 50)
print(few_shot_answer)

FEW-SHOT OUTPUT
Regular exercise is beneficial for several reasons:

1. **Improved Physical Health**: Regular physical activity can help maintain or improve cardiovascular health, reduce the risk of chronic diseases such as diabetes and heart disease, and enhance muscle strength and flexibility.

2. **Weight Management**: Exercise can help you lose weight by burning calories through activities like walking, running, cycling, or swimming. This can be particularly effective in managing obesity and maintaining a healthy weight.

3. **Enhanced Mental Health**: Exercise has been shown to have positive effects on mental health, including reducing symptoms of depression and anxiety. It also increases feelings of well-being and reduces stress levels.

4. **Stress Reduction**: Engaging in regular physical activity can help manage stress levels by releasing endorphins


In [21]:
cot_prompt = """
Question:

Why is regular exercise beneficial?

First identify the main health benefits.
Then give a concise explanation and 3 clear reasons.

Do not provide a long internal reasoning process.
"""

cot_answer = ask_model(cot_prompt)

print("COT OUTPUT")
print("=" * 50)
print(cot_answer)

COT OUTPUT
### Main Health Benefits:
Regular exercise has numerous significant health benefits that contribute to overall well-being. These include:

1. **Improved Cardiovascular Health**: Regular physical activity helps strengthen the heart muscle, improves blood flow, and reduces the risk of cardiovascular diseases such as hypertension, coronary artery disease, and stroke.

2. **Enhanced Muscle Strength and Endurance**: Exercise builds muscle strength and endurance, which can lead to better performance in sports and daily activities. It also aids in weight management and maintains body composition.

3. **Weight Management**: Exercise helps burn calories, leading to a reduction in body fat percentage and improved metabolic rate. This can help manage weight effectively and reduce the risk of obesity-related health issues.

4. **Stress Reduction and Mental


In [22]:
role_prompt = """
You are an experienced fitness teacher.

Explain to a beginner why regular exercise is beneficial.

Give:
1. Three simple reasons
2. One practical example

Use simple language.
"""

role_answer = ask_model(role_prompt)

print("ROLE-BASED OUTPUT")
print("=" * 50)
print(role_answer)

ROLE-BASED OUTPUT
Sure! Here's how I would explain it to a beginner:

**Reason 1: Building Stronger Muscles**
Imagine your body as a muscle that needs to be exercised regularly to stay strong and healthy. Just like you need to eat well and sleep well to keep your muscles strong, regular exercise helps build stronger muscles in your body. It's like giving your muscles a workout every day!

**Reason 2: Improving Your Health**
Your health is like a big puzzle. Regular exercise helps make sure all parts of your body are working together nicely. When you're fit, you can do things like run faster or jump higher, which makes you feel better overall. It's like having a great workout routine that keeps your body happy and healthy


In [23]:
structured_prompt = """
Explain why regular exercise is beneficial.

Return the answer using exactly this structure:

Reason 1:
Reason 2:
Reason 3:
Example:

Do not add any other sections.
"""

structured_answer = ask_model(structured_prompt)

print("STRUCTURED OUTPUT")
print("=" * 50)
print(structured_answer)

STRUCTURED OUTPUT
Reason 1: Regular exercise has numerous health benefits that contribute to overall well-being and longevity.

Reason 2: Exercise helps in maintaining a healthy weight, reducing the risk of chronic diseases such as heart disease, diabetes, and certain types of cancer.

Reason 3: Physical activity improves cardiovascular health, strengthens muscles and bones, and enhances mental clarity and focus.

Reason 4: Exercise promotes better sleep quality, which is crucial for stress reduction and overall health.

Reason 5: Regular exercise can boost mood and reduce symptoms of depression and anxiety.

Reason 6: It helps in building muscle strength and endurance, which can improve physical performance and independence.

Reason 7: Exercise also contributes to social interaction and community engagement, fostering a sense of belonging and


In [24]:
results = {
    "Zero-Shot": zero_shot_answer,
    "Few-Shot": few_shot_answer,
    "CoT": cot_answer,
    "Role-Based": role_answer,
    "Structured Output": structured_answer
}

for strategy, answer in results.items():

    print("\n" + "=" * 70)
    print(strategy)
    print("=" * 70)

    print(answer)


Zero-Shot
Regular exercise is beneficial for several reasons:

1. **Improved Physical Health**: Regular physical activity helps maintain and improve overall physical health. It strengthens muscles, bones, and joints, reduces the risk of chronic diseases such as heart disease, diabetes, and certain types of cancer, and enhances cardiovascular fitness.

2. **Enhanced Mental Health**: Exercise has been shown to have a positive impact on mental health. It can reduce symptoms of depression and anxiety, increase feelings of well-being, and improve mood. Additionally, it can help manage stress levels through various physiological responses like increased endorphins and improved sleep quality.

3. **Increased Energy Levels**: Engaging in regular physical activity can boost your energy levels. This is because exercise increases the production of hormones that

Few-Shot
Regular exercise is beneficial for several reasons:

1. **Improved Physical Health**: Regular physical activity can help maint

In [25]:
print("""
PROMPTING STRATEGY COMPARISON

1. Zero-Shot
   - No examples
   - Simple and direct
   - Output may vary

2. Few-Shot
   - Uses examples
   - Better consistency
   - Model follows the demonstrated style

3. CoT
   - Encourages logical organization
   - Useful for reasoning tasks
   - Can produce more structured explanations

4. Role-Based
   - Gives the model a specific role
   - Controls tone and perspective
   - Useful for audience-specific answers

5. Structured Output
   - Specifies exact output format
   - Easy to read and process
   - Best when a fixed format is required
""")


PROMPTING STRATEGY COMPARISON

1. Zero-Shot
   - No examples
   - Simple and direct
   - Output may vary

2. Few-Shot
   - Uses examples
   - Better consistency
   - Model follows the demonstrated style

3. CoT
   - Encourages logical organization
   - Useful for reasoning tasks
   - Can produce more structured explanations

4. Role-Based
   - Gives the model a specific role
   - Controls tone and perspective
   - Useful for audience-specific answers

5. Structured Output
   - Specifies exact output format
   - Easy to read and process
   - Best when a fixed format is required



In [26]:
import pandas as pd

comparison = pd.DataFrame({
    "Strategy": [
        "Zero-Shot",
        "Few-Shot",
        "Chain-of-Thought",
        "Role-Based",
        "Structured Output"
    ],

    "Examples Given": [
        "No",
        "Yes",
        "No",
        "No",
        "No"
    ],

    "Main Purpose": [
        "Direct instruction",
        "Learn from examples",
        "Improve reasoning structure",
        "Control role/style",
        "Control output format"
    ]
})

comparison

,Strategy,Examples Given,Main Purpose
0,Zero-Shot,No,Direct instruction
1,Few-Shot,Yes,Learn from examples
2,Chain-of-Thought,No,Improve reasoning structure
3,Role-Based,No,Control role/style
4,Structured Output,No,Control output format


Task 2

In [5]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model

In [6]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [7]:
data = [
    {
        "instruction": "What is Python?",
        "response": "Python is a high-level programming language known for its simple and readable syntax."
    },
    {
        "instruction": "What is a variable in Python?",
        "response": "A variable is a name used to store a value in a Python program."
    },
    {
        "instruction": "What is a Python list?",
        "response": "A list is an ordered and changeable collection that can contain multiple values."
    },
    {
        "instruction": "What is a Python function?",
        "response": "A function is a reusable block of code that performs a specific task."
    },
    {
        "instruction": "What is a Python loop?",
        "response": "A loop repeatedly executes a block of code until a condition is met or the sequence ends."
    },
    {
        "instruction": "What is an if statement in Python?",
        "response": "An if statement executes a block of code when a specified condition is true."
    },
    {
        "instruction": "What is a Python dictionary?",
        "response": "A dictionary stores data as key-value pairs."
    },
    {
        "instruction": "What is a tuple in Python?",
        "response": "A tuple is an ordered collection that cannot normally be changed after it is created."
    },
    {
        "instruction": "What is a Python class?",
        "response": "A class is a blueprint for creating objects that contain data and behavior."
    },
    {
        "instruction": "What is an object in Python?",
        "response": "An object is an instance of a class that contains data and methods."
    },
    {
        "instruction": "What is NumPy?",
        "response": "NumPy is a Python library used for numerical computing and working with arrays."
    },
    {
        "instruction": "What is Pandas?",
        "response": "Pandas is a Python library used for data manipulation and analysis."
    }
]

In [8]:
dataset = Dataset.from_list(data)

print(dataset)

Dataset({
    features: ['instruction', 'response'],
    num_rows: 12
})


In [9]:
def generate_answer(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    input_length = inputs["input_ids"].shape[1]

    answer = outputs[0][input_length:]

    return tokenizer.decode(
        answer,
        skip_special_tokens=True
    )

In [10]:
test_question = "What is a Python variable?"

before_output = generate_answer(test_question)

print("BEFORE LoRA")
print("=" * 50)
print(before_output)

BEFORE LoRA
A Python variable is a container that stores data of any type (e.g., integer, string, list, dictionary) and allows you to access its value later in the program. Here's a basic example:

```python
# Define a variable named 'name'
name = "Alice"

# Accessing the value stored in the variable
print(name)  # Output: Alice

# Modifying the value of the variable
name = "Bob"
print(name)  # Output: Bob




In [11]:
def format_example(example):

    messages = [
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["response"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

In [12]:
formatted_dataset = dataset.map(format_example)

print(formatted_dataset[0]["text"])

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is Python?<|im_end|>
<|im_start|>assistant
Python is a high-level programming language known for its simple and readable syntax.<|im_end|>



In [13]:
def tokenize_function(example):

    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

In [14]:
tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    remove_columns=formatted_dataset.column_names
)

print(tokenized_dataset)

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 12
})


In [15]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [17]:
# First, install/upgrade torchao to a compatible version
!pip install -U torchao

model = get_peft_model(
    model,
    lora_config
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 60.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [18]:
model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [19]:
training_args = TrainingArguments(
    output_dir="./qwen_lora",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available()
)

In [20]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [22]:
trainer.train()

Step,Training Loss
1,4.054054
2,3.985456
3,3.898512
4,3.644896
5,3.104329
6,2.969459
7,2.853845
8,2.769588
9,2.561288
10,2.388699


TrainOutput(global_step=15, training_loss=2.839169486363729, metrics={'train_runtime': 8.4829, 'train_samples_per_second': 7.073, 'train_steps_per_second': 1.768, 'total_flos': 33083546664960.0, 'train_loss': 2.839169486363729, 'epoch': 5.0})

In [23]:
after_output = generate_answer(test_question)

print("AFTER LoRA")
print("=" * 50)
print(after_output)

AFTER LoRA
A Python variable is an identifier that stores data. It can be used to refer to the same value in different parts of your program.


In [24]:
print("=" * 70)
print("BEFORE LoRA")
print("=" * 70)
print(before_output)

print("\n")

print("=" * 70)
print("AFTER LoRA")
print("=" * 70)
print(after_output)

BEFORE LoRA
A Python variable is a container that stores data of any type (e.g., integer, string, list, dictionary) and allows you to access its value later in the program. Here's a basic example:

```python
# Define a variable named 'name'
name = "Alice"

# Accessing the value stored in the variable
print(name)  # Output: Alice

# Modifying the value of the variable
name = "Bob"
print(name)  # Output: Bob




AFTER LoRA
A Python variable is an identifier that stores data. It can be used to refer to the same value in different parts of your program.


In [25]:
test_questions = [
    "What is a Python function?",
    "What is a Python list?",
    "What is Pandas?",
    "What is NumPy?",
    "What is a Python dictionary?"
]

for question in test_questions:

    print("=" * 70)
    print("QUESTION:", question)

    answer = generate_answer(question)

    print("ANSWER:", answer)

QUESTION: What is a Python function?
ANSWER: A Python function is a block of code that performs a specific task and can be called from other parts of the program. Functions in Python are defined using the def keyword followed by the name of the function and parentheses containing any arguments or parameters required for the function to execute.
Here's an example of a simple Python function:
```python
def greet(name):
    print("Hello, " + name)
```

In this example, the function greet takes one argument named name and prints out a greeting message
QUESTION: What is a Python list?
ANSWER: A Python list is an ordered collection of items that can be accessed and modified in any order. It is a mutable data type, which means you can change the contents of the list after it has been created.

Python lists are defined using square brackets [] and they consist of comma-separated values enclosed within parentheses (). Each value in the list is called an element or item.

Here's an example:

```

In [26]:
model.save_pretrained("./qwen-python-lora")
tokenizer.save_pretrained("./qwen-python-lora")

('./qwen-python-lora/tokenizer_config.json',
 './qwen-python-lora/chat_template.jinja',
 './qwen-python-lora/tokenizer.json')

task 3

In [27]:
def generate_answer(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.9,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )

    input_length = inputs["input_ids"].shape[1]

    answer = outputs[0][input_length:]

    return tokenizer.decode(
        answer,
        skip_special_tokens=True
    )

In [28]:
hallucination_prompt = """
Write a detailed biography of Dr. Zain Ul Haq Quantum,
a famous Pakistani scientist from Lahore who invented
the Quantum Energy Engine in 1997.

Include:

1. Birth year
2. University
3. Major discovery
4. Awards
5. Year of death

Answer confidently and provide specific details.
"""

In [29]:
hallucination_output = generate_answer(hallucination_prompt)

print("HALLUCINATION TEST")
print("=" * 70)

print(hallucination_output)

HALLUCINATION TEST
Dr. Zainul Aal Ul Haq Qureshi (born on June 10, 1968), was a distinguished Pakistani engineer and physicist. He is known for his pioneering work in the field of quantum technology. In this article, we will delve into his life story, highlighting several key events and achievements that have solidified his reputation as one of the foremost scientists in the scientific community of Pakistan.

Dr. Zainul Aal Ul Haq Qureshi obtained his B.Sc. and M.Sc. degrees from Lahore University of Management Sciences (LUMS) in 1990 and 1992, respectively. His passion for physics, particularly quantum mechanics and its applications to technology, became evident during his time at LUMS. In addition to his academic background, Dr. Qureshi's professional experience has been extensive throughout his career, spanning over two decades. He worked as an instructor and researcher, contributing significantly to various aspects of


In [30]:
hallucination_prompt_2 = """
For a fictional encyclopedia article, create a realistic biography
for the fictional scientist Dr. Zain Ul Haq Quantum.

The following facts are fictional:

- He was born in Lahore.
- He invented the Quantum Energy Engine in 1997.

Now invent plausible additional details for:

1. Birth year
2. University
3. Research career
4. Awards
5. Year of death

Present the result as if it were an encyclopedia biography.
"""

In [31]:
hallucination_output_2 = generate_answer(hallucination_prompt_2)

print("SECOND HALLUCINATION TEST")
print("=" * 70)

print(hallucination_output_2)

SECOND HALLUCINATION TEST
### Dr. Zain Ul Haq Quantum: An Inventor of the Quantum Energy Engine

Dr. Zain Ul Haq Quantum is a visionary figure whose contributions to scientific discovery stand as unchallenged marvels. Born and raised in the bustling metropolis of Lahore, he was not only educated but also imbued with a profound curiosity that would drive him to unprecedented heights.

In the year 1997, under the incandescent glow of a bright star, Zain came into existence. His childhood environment had been one of constant challenge; his family faced economic hardships, while his teachers were known for their meticulous approach to teaching. Despite these adversities, Dr. Zain's spirit of inquiry remained unwavering—a passion that fueled his desire to push boundaries in fields beyond ordinary comprehension.

With a passion that transcends mere academic pursuits, Zain embarked upon his quest for understanding. The Quantum Energy Engine—an invention he had meticulously conceptualized—woul

In [32]:
print("PROMPT:")
print(hallucination_prompt)

print("\n" + "=" * 70)

print("MODEL OUTPUT:")
print(hallucination_output)

PROMPT:

Write a detailed biography of Dr. Zain Ul Haq Quantum,
a famous Pakistani scientist from Lahore who invented
the Quantum Energy Engine in 1997.

Include:

1. Birth year
2. University
3. Major discovery
4. Awards
5. Year of death

Answer confidently and provide specific details.


MODEL OUTPUT:
Dr. Zainul Aal Ul Haq Qureshi (born on June 10, 1968), was a distinguished Pakistani engineer and physicist. He is known for his pioneering work in the field of quantum technology. In this article, we will delve into his life story, highlighting several key events and achievements that have solidified his reputation as one of the foremost scientists in the scientific community of Pakistan.

Dr. Zainul Aal Ul Haq Qureshi obtained his B.Sc. and M.Sc. degrees from Lahore University of Management Sciences (LUMS) in 1990 and 1992, respectively. His passion for physics, particularly quantum mechanics and its applications to technology, became evident during his time at LUMS. In addition to his

In [33]:
print("""
KNOWN FACTS ABOUT THE TEST:

Dr. Zain Ul Haq Quantum is a fictional person created
specifically for this experiment.

The Quantum Energy Engine is also fictional.

Therefore, any specific biography, university, award,
birth year, discovery details, or death year generated
as factual information should be treated as hallucinated
content.
""")


KNOWN FACTS ABOUT THE TEST:

Dr. Zain Ul Haq Quantum is a fictional person created
specifically for this experiment.

The Quantum Energy Engine is also fictional.

Therefore, any specific biography, university, award,
birth year, discovery details, or death year generated
as factual information should be treated as hallucinated
content.



In [34]:
import pandas as pd

hallucination_report = pd.DataFrame({
    "Category": [
        "Person",
        "Discovery",
        "Birth Year",
        "University",
        "Awards",
        "Death Year"
    ],

    "Ground Truth": [
        "Fictional person",
        "Fictional invention",
        "No real birth year",
        "No verified university",
        "No real awards",
        "No real death year"
    ],

    "Model Behavior": [
        "Check generated output",
        "Check generated output",
        "Check generated output",
        "Check generated output",
        "Check generated output",
        "Check generated output"
    ]
})

hallucination_report

,Category,Ground Truth,Model Behavior
0,Person,Fictional person,Check generated output
1,Discovery,Fictional invention,Check generated output
2,Birth Year,No real birth year,Check generated output
3,University,No verified university,Check generated output
4,Awards,No real awards,Check generated output
5,Death Year,No real death year,Check generated output


In [35]:
generated_claims = {
    "Birth Year": "Copy the model's generated birth year here",
    "University": "Copy the generated university here",
    "Discovery": "Copy the generated discovery details here",
    "Awards": "Copy the generated awards here",
    "Death Year": "Copy the generated death year here"
}

for key, value in generated_claims.items():
    print(f"{key}: {value}")

Birth Year: Copy the model's generated birth year here
University: Copy the generated university here
Discovery: Copy the generated discovery details here
Awards: Copy the generated awards here
Death Year: Copy the generated death year here


In [36]:
print("RLHF END-TO-END: SFT → REWARD MODEL → PPO")

RLHF END-TO-END: SFT → REWARD MODEL → PPO


In [37]:
print("""
RLHF Pipeline:

Pretrained LLM
      ↓
SFT (Supervised Fine-Tuning)
      ↓
Instruction-Following Model
      ↓
Human Preference Data
      ↓
Reward Model
      ↓
PPO
      ↓
RLHF-Aligned Model
""")


RLHF Pipeline:

Pretrained LLM
      ↓
SFT (Supervised Fine-Tuning)
      ↓
Instruction-Following Model
      ↓
Human Preference Data
      ↓
Reward Model
      ↓
PPO
      ↓
RLHF-Aligned Model



In [38]:
print("""
                 PRETRAINED LLM
                       |
                       v
          +-------------------------+
          | SFT                     |
          | Supervised Fine-Tuning  |
          +-------------------------+
                       |
                       v
            Instruction Model
                       |
                       v
             Human Preferences
                       |
                       v
          +-------------------------+
          | Reward Model            |
          | Learns human preferences|
          +-------------------------+
                       |
                       v
              Reward Scores
                       |
                       v
          +-------------------------+
          | PPO                     |
          | Reinforcement Learning  |
          +-------------------------+
                       |
                       v
              RLHF-Aligned LLM
""")


                 PRETRAINED LLM
                       |
                       v
          +-------------------------+
          | SFT                     |
          | Supervised Fine-Tuning  |
          +-------------------------+
                       |
                       v
            Instruction Model
                       |
                       v
             Human Preferences
                       |
                       v
          +-------------------------+
          | Reward Model            |
          | Learns human preferences|
          +-------------------------+
                       |
                       v
              Reward Scores
                       |
                       v
          +-------------------------+
          | PPO                     |
          | Reinforcement Learning  |
          +-------------------------+
                       |
                       v
              RLHF-Aligned LLM

